# CH1 Canonical Pipeline -- Driver Notebook

Thin, self-verifying driver over the tested `pipeline.*` package. Every cell below
is an `import` + one call into `pipeline.*` + a `display()`/`assert` of the result
-- **no heavy logic is reimplemented here**. This notebook is a narrated table of
contents over Phases 0-5 of the canonical-pipeline rebuild (`pipeline/config.py`
through `pipeline/report.py`), not a reimplementation of any of them.

Every stage's on-disk artifacts already exist from prior real runs (`data/raw/`,
4.6 GB, plus every intermediate `data/processed/**` cache), and every pipeline
module is cache-aware, so most cells below are cache-served (fast reads / cheap
re-scoring) rather than cold refits. The one deliberate exception is the 5 km
spatial-CV diagnostic in Stage 4, which is a genuinely different fold partition
and is routed to its own output directory so it can never clobber the 2 km run's
own cache.

**Stages:**
0. Foundation (`pipeline.config`, `pipeline.qa`)
1. Presence & predictors (`pipeline.hwm`, `pipeline.predictors`)
2. Screening (`pipeline.screen`)
3. MaxEnt (`pipeline.maxent`)
4. Evaluation (`pipeline.evaluate`, `pipeline.model_compare`, `pipeline.robustness`)
5. Sensitivity & equity (`pipeline.sensitivity`, `pipeline.equity`)
6. Report (`pipeline.report`)


In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# The kernel's cwd is this notebook's own directory (notebooks/), not the repo
# root, so the top-level `pipeline` package (not pip-installed -- see
# environment.yml) is not importable by default. Walk upward from cwd to find
# the directory containing pipeline/__init__.py and put it on sys.path -- robust
# to whether this notebook is executed via nbconvert (cwd = notebooks/) or run
# interactively from a Jupyter server started at the repo root (cwd = repo root).
def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pipeline" / "__init__.py").exists():
            return candidate
    raise RuntimeError(f"could not locate the repo root (pipeline/__init__.py) from {start}")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"repo root on sys.path: {REPO_ROOT}")

import pandas as pd
from IPython.display import display, Image

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## Stage 0 -- Foundation

Global seed, the canonical train/test split, spatial-CV blocking, and run
provenance -- the invariants every later stage is built on.


In [ ]:
from pipeline import config, qa

print("GLOBAL_SEED:", config.GLOBAL_SEED)

samples = config.canonical_samples()
train_idx, test_idx = config.split(samples)
print(f"canonical_samples(): {len(samples)} points -> split(): {len(train_idx)} train / {len(test_idx)} test")

qa.assert_no_leakage(train_idx, test_idx)
qa.assert_counts(canonical=(277, len(samples)), train=(194, len(train_idx)), test=(83, len(test_idx)))
print("gate: train/test disjoint; canonical/train/test sizes = 277/194/83 -- OK")

fold_ids = config.spatial_folds(samples, block_km=2)
n_blocks = len(set(fold_ids.tolist()))
print(f"spatial_folds(block_km=2): {n_blocks} raw 2 km blocks over the canonical set")

prov = config.provenance()
display(prov)
assert {"git_sha", "config_hash", "python", "created"} <= set(prov)
print("gate: provenance keys present -- OK")


## Stage 1 -- Presence & predictors

Flood-observation inventory (`pipeline.hwm`) and the aligned 22-layer predictor
stack + QA gates (`pipeline.predictors`) -- the D8-routing fix shows up
directly in `flow_accumulation`'s max.


In [ ]:
from pipeline import hwm

raw = hwm.raw_counts()
display(raw)
assert raw["HW_raw"] == 595
print("gate: raw HW count = 595 -- OK")

presence = hwm.build_presence()
n_hw = int((presence["type"] == "HW").sum())
n_eow = int((presence["type"] == "EOW").sum())
print(f"build_presence(): {len(presence)} points after 30 m thinning ({n_hw} HW + {n_eow} EOW)")
assert len(presence) == 286 and n_hw == 279 and n_eow == 7
print("gate: 595 raw HW -> 286 thinned (279 HW + 7 EOW) -- OK")


In [ ]:
import numpy as np
from pipeline import predictors

# acquire_all() returns the 18 terrain/soils/land-use/census layers (never the
# 4 D8-routing hydrology layers -- those come from the separate flow_derivatives()
# call below, mirroring tests/test_predictors_align.py's own completeness check).
stack = predictors.acquire_all()
qa.assert_aligned(list(stack.values()))
assert set(stack.keys()) == set(predictors.EXPECTED_NON_CENSUS) | set(predictors.CENSUS_LAYERS)
print(f"acquire_all(): {len(stack)} aligned terrain/soils/land-use/census layers -- OK")

# The D8-routing fix (mirrors tests/test_routing.py::test_real_aoi_reaches_river_scale
# exactly): flow_derivatives() is cache-aware, so this reuses the already-computed
# flow_accumulation/twi/spi/hand rasters rather than re-running WhiteboxTools.
flow = predictors.flow_derivatives(predictors.dem_path())
qa.assert_aligned(list(stack.values()) + [flow["flow_accumulation"]])
fa_max = float(np.nanmax(flow["flow_accumulation"].values))
print(f"flow_derivatives(): flow_accumulation max = {fa_max:,.0f} "
      "(the D8-routing fix; the old, broken xarray-spatial chain capped this at 2,451)")
assert fa_max > 50_000
print("gate: acquire_all() + flow_derivatives() together align to 22 raw layers; routing fix confirmed -- OK")

valid = predictors.valid_mask()
print(f"valid_mask(): {int(valid.sum())} / {len(valid)} presence points pass the predictor-NaN gate")
assert int(valid.sum()) == 279

qa_rep = predictors.qa_report()
display(pd.Series(qa_rep["layer_counts"]))
display(pd.Series(qa_rep["presence_points"]))
print("gate: qa_report() layer-count + presence-point accounting -- OK")


## Stage 2 -- Screening

Two-stage multicollinearity screen (Pearson |r| >= 0.8, then VIF > 10) and the
presence-vs-background bivariate characterization -- explicitly labeled NOT
validation.


In [ ]:
from pipeline import screen

raw_stack = screen._load_stack()
kept, mc_report = screen.multicollinearity(raw_stack)
dropped = sorted(mc_report.index[mc_report["dropped"]].tolist())
print(f"multicollinearity(): {len(raw_stack.columns)} raw predictors -> {len(kept)} kept, dropped = {dropped}")
assert dropped == sorted(["tri", "tpi", "flow_accumulation", "spi"])
assert "twi" in kept and "flow_accumulation" not in kept
print("gate: tri/tpi dropped at Stage 1 (Pearson); TWI retained over flow_accumulation at Stage 2 (VIF) -- OK")


In [ ]:
biv = screen.bivariate(presence)
display(biv[["predictor", "type", "test", "effect", "effect_metric", "p_bonferroni", "significant"]])

assert biv.attrs["kind"] == "characterization" and biv.attrs["not_validation"] is True
assert biv.attrs["n_tests"] == len(kept) == len(biv)
print(f"gate: bivariate() characterizes all {biv.attrs['n_tests']} screened predictors, framed as NOT validation -- OK")


## Stage 3 -- MaxEnt

AICc tuning grid, the locked publication config, the final (10-replicate
bootstrap) and evaluation (train-only) model families, and combined variable
importance.


In [ ]:
from pipeline import maxent

modeling = maxent.modeling_layers()
print(f"modeling_layers(): {len(modeling)} predictors (median_income dropped from the {len(kept)} screened); TWI retained: {'twi' in modeling}")
assert len(modeling) == 17 and "twi" in modeling and "median_income" not in modeling
print("gate: 17-predictor canonical modeling stack -- OK")

grid = maxent.aicc_grid()
display(grid)

best = maxent.select_best(grid)
print(f"select_best(grid): raw AICc winner beta={best['beta']} features={best['feature_classes']} k={best['k']} AICc={best['AICc']:.2f}")
print(f"FINAL_CONFIG (locked, owner's interior-optimum choice -- see maxent.py docstring): {maxent.FINAL_CONFIG}")
assert maxent.FINAL_CONFIG == {"beta": 0.5, "features": "LQ"}
print("gate: published/evaluation model family locked to beta=0.5, LQ -- OK")


In [ ]:
# The published raster is a 10-replicate MaxEnt bootstrap ensemble fit with no
# fixed CLI seed (see maxent.fit_final's docstring), so refitting on every run
# would regenerate the canonical map with small run-to-run jitter and needlessly
# perturb Stage 5's equity zonal stats. Reuse the already-published canonical
# raster when present (consistent with this notebook's cache-aware design); only
# fit when it is genuinely absent. maxent.FINAL_DIR is fit_final()'s own default
# out_dir; the cloglog mean map lives under it (== evaluate.FINAL_RASTER).
canonical_raster = maxent.FINAL_DIR / "cloglog" / "flood_avg.tif"
if canonical_raster.exists():
    final_raster = canonical_raster
    print(f"reusing cached published raster (bootstrap ensemble is non-deterministic; "
          f"reuse keeps the map + downstream numbers stable): {final_raster}")
else:
    final_raster = maxent.fit_final()
    print(f"fit_final() -> {final_raster}")
apparent = maxent.apparent_auc()
print(f"apparent_auc() [resubstitution -- NEVER the headline number]: {apparent['mean']:.4f} +/- {apparent['sd']:.4f} ({apparent['n_replicates']} replicates)")

# The evaluation model IS deterministic (replicates=0) and Invariant-2 critical
# -- always (re)fit it; never reuse-guard this path.
eval_raster = maxent.fit_eval(train_idx)
qa.assert_eval_model_excludes_test(train_idx, test_idx)
print(f"fit_eval(train_idx) -> {eval_raster}")
print("gate: evaluation model's train_idx excludes every test point (Invariant 2) -- OK")

vi = maxent.variable_importance_combined()
display(vi)
top_predictor = vi.sort_values("unique_contribution", ascending=False).iloc[0]["predictor"]
assert top_predictor == "distance_to_outfall"
print(f"gate: top predictor by jackknife unique contribution = {top_predictor} -- OK")

## Stage 4 -- Evaluation

The honest out-of-sample AUC, the terrain-only GFI baseline (on the
fixed routing), the 6-model comparison with a corrected significance test,
spatial-block / leave-one-event-out cross-validation, predictor-group
ablation, and directional (never "validated against") external comparisons.


In [ ]:
from pipeline import evaluate

test_pts = samples.iloc[test_idx][["x", "y"]]
background = evaluate.sample_background(evaluate.EVAL_RASTER, n=evaluate.DEFAULT_BACKGROUND_N, seed=config.GLOBAL_SEED)
oos = evaluate.oos_auc(evaluate.EVAL_RASTER, test_pts, train_idx, test_idx, background)
print(f"oos_auc(): honest out-of-sample AUC = {oos:.4f}")
assert 0.95 < oos < 1.0 and abs(oos - 0.982577108433735) < 0.005
print("gate: honest OOS AUC ~= 0.983, sampled from the TRAIN-ONLY eval raster, never FINAL_RASTER -- OK")

gfi_result = evaluate.gfi_baseline()
print(f"gfi_baseline(): terrain-only AUC = {gfi_result['auc']:.4f} (flow_accumulation_max = {gfi_result['flow_accumulation_max']:,.0f})")
assert gfi_result["flow_accumulation_max"] > 50_000
assert abs(gfi_result["auc"] - 0.47) < 0.02
print("gate: terrain alone cannot discriminate (AUC ~= 0.47), computed on the FIXED D8 routing -- OK")


In [ ]:
from pipeline import model_compare, robustness

mc_background = model_compare.build_background()
comparison = model_compare.run(train_idx, test_idx, mc_background)
display(comparison[["model", "auc", "sensitivity", "specificity", "tss", "f1"]])

expected_models = {"MaxEnt (reference)", "Logistic Regression", "Random Forest", "XGBoost", "LightGBM", "CatBoost"}
assert set(comparison["model"]) == expected_models
ranking = comparison.set_index("model")["auc"].sort_values(ascending=False)
print("AUC ranking (highest first):", list(ranking.index))
print("gate: 6-model comparison on the identical, matched test set, no test-set early-stopping -- OK")

sig = robustness.loeo_significance_test()
print(f"loeo_significance_test(): MaxEnt vs {sig['second_model']} across {sig['k']} LOEO folds -- "
      f"mean_diff={sig['mean_diff']:+.4f}, Nadeau-Bengio p={sig['p_value']:.3f}, verdict={sig['verdict']}")
assert sig["verdict"] == "within_noise"
print("gate: MaxEnt ~= XGBoost under the Nadeau-Bengio corrected test (p ~= 0.53), not a bare Wilcoxon-at-p-floor -- OK")


In [ ]:
rasters = model_compare.susceptibility_rasters(train_idx, test_idx, mc_background, run_results=comparison)
print(f"susceptibility_rasters(): wrote {len(rasters)} per-model rasters under {model_compare.RASTER_DIR}")
for label, path in rasters.items():
    print(f"  {label}: {path.name}")

assert set(rasters) == set(model_compare.ML_MODEL_LABELS)
assert all(p.exists() and p.stat().st_size > 0 for p in rasters.values())
print("gate: full-AOI susceptibility raster written per ML comparator (Logistic Regression/"
      "Random Forest/XGBoost/LightGBM/CatBoost), cache-aware; the consistency-AUC guard held "
      "for every model against model_compare.run()'s own reported AUC (see the '... consistency "
      "OK' lines printed above -- a mismatch there would have raised, not printed) -- OK")


In [ ]:
cv2 = robustness.spatial_cv(block_km=2)
print(f"spatial_cv(block_km=2): mean AUC = {cv2['mean_auc']:.4f} +/- {cv2['sd_auc']:.4f} "
      f"({cv2['n_folds']} folds, merged from {cv2['n_raw_blocks']} raw blocks)")

# A 5 km grouping is a genuinely DIFFERENT fold partition (fewer, larger blocks) --
# routed to its own out_dir so it can never clobber the 2 km run's own cached fold
# rasters (and the pytest suite's own pinned regression values) above.
cv5 = robustness.spatial_cv(block_km=5, out_dir=robustness.SPATIAL_CV_DIR.parent / "spatial_cv_5km")
print(f"spatial_cv(block_km=5): mean AUC = {cv5['mean_auc']:.4f} +/- {cv5['sd_auc']:.4f} "
      f"({cv5['n_folds']} folds, merged from {cv5['n_raw_blocks']} raw blocks)")
assert cv2["mean_auc"] > 0.9 and cv5["mean_auc"] > 0.9
print("gate: spatial block-CV -- the honest, spatially-robust generalization AUC at two block sizes -- OK")

loeo_result = robustness.loeo()
print(f"loeo(): mean AUC = {loeo_result['mean_auc']:.4f} +/- {loeo_result['sd_auc']:.4f} ({loeo_result['n_events']} events)")

ablation_df = robustness.ablation()
display(ablation_df)
baseline_auc = float(ablation_df.loc[ablation_df["arm"] == "baseline", "auc"].iloc[0])
assert (ablation_df["auc"] <= baseline_auc + 1e-9).all()
print("gate: ablation baseline reuses the evaluate.oos_auc protocol bit-for-bit; every arm's AUC <= baseline -- OK")


In [ ]:
ext = evaluate.external_comparisons()
print(f"external_comparisons(): threshold = {ext['threshold']:.4f}")
for product in ("fema", "nws", "s1"):
    print(f"  {product}: kind={ext[product]['kind']} status={ext[product].get('status', 'ok')}")

assert ext["kind"] == "directional_consistency"
framing = ext["framing"].lower()
assert "not" in framing and "validation" in framing
print("gate: FEMA/NWS/Sentinel-1 comparisons framed as DIRECTIONAL CONSISTENCY, never 'validated against' -- OK")


## Stage 5 -- Sensitivity & equity

Matched-baseline sensitivity arms (the ascertainment-bias bound) and the
corrected ACS equity / environmental-justice overlay (Holm/FDR-corrected
Spearman family).


In [ ]:
from pipeline import sensitivity

baseline, arms = sensitivity.run()
arms_df = pd.DataFrame(arms)
display(arms_df[["arm", "description", "auc", "baseline_auc", "delta_auc"]])

outfall_matched = next(a for a in arms if a["arm"] == "bg_outfall_distance_matched")
print(f"ascertainment bound: baseline AUC={baseline['auc']:.4f} -> outfall-distance-matched AUC={outfall_matched['auc']:.4f}")
assert abs(baseline["auc"] - 0.9826) < 1e-3
assert abs(outfall_matched["auc"] - 0.9294) < 1e-3
print("gate: baseline 0.9826 -> outfall-distance-matched 0.9294 -- real signal beyond outfall proximity survives -- OK")


In [ ]:
from pipeline import equity

eq = equity.run()
corr = eq["correlations"]
display(corr[["variable", "rho", "p_raw", "p_adjusted", "p_adjusted_fdr_bh", "significant_raw", "significant_corrected"]])

survivors = corr.loc[corr["significant_corrected"], "variable"].tolist()
pop_rho = float(corr.loc[corr["variable"] == "bg_pop_density_mean", "rho"].iloc[0])
demographic_rows = corr[~corr["coupled_with_predictor"]]

print(f"only {survivors} survives Holm/FDR correction (rho={pop_rho:.3f}); "
      f"no genuine sociodemographic indicator (incl. income) reaches significance")
# equity.run() reads the published susceptibility raster, a 10-replicate MaxEnt
# bootstrap ensemble with no fixed CLI seed (maxent.fit_final's docstring), so
# pop_rho carries small run-to-run jitter (observed ~0.61-0.64). Assert the
# jitter-ROBUST invariants -- the survivor SET, a clearly-positive rho well
# above the significance neighborhood, and NO demographic indicator surviving --
# never an exact rho pin (which the previous abs(pop_rho-0.63)<0.02 gate was).
assert survivors == ["bg_pop_density_mean"]
assert pop_rho > 0.5
assert not demographic_rows["significant_corrected"].any()
print("gate: only the mechanically-coupled population_density indicator survives correction -- OK")

## Stage 6 -- Report

The paper's presentation layer (`pipeline.report`) -- Tables 1-4 (including
the predictor-stack Table 2) plus the full paper figure suite (300 DPI
PNG + vector PDF, publication-ready), assembled entirely by reading artifacts
the stages above already produced.


In [ ]:
from pipeline import report

tbls = report.tables()
for name in ("table1", "table2", "table3", "table4"):
    print(f"--- {name} ---")
    display(tbls[name])

assert tbls["table1"].attrs["raw_hw"] == 595 and tbls["table1"].attrs["eow_status"] == "used"
assert tbls["table2"].attrs["n_modeled"] == 17 and tbls["table2"].attrs["n_considered"] == 24
assert tbls["table3"].iloc[0]["predictor"] == "distance_to_outfall"
required_arms = {"bg_target_group_stormwater_buffer", "bg_outfall_distance_matched"}
assert required_arms <= set(tbls["table4"]["Arm"])
print(f"gate: Tables 1-4 (+ {len(tbls) - 4} nice-to-have) written to reports/tables/ (CSV+LaTeX) -- OK")


In [ ]:
figs = report.figures()
pdfs = [p.with_suffix(".pdf") for p in figs]
print(f"{len(figs)} figures written to {report.FIGURES_DIR}:")
for p, pdf in zip(figs, pdfs):
    print(f"  {p.name}  (+ {pdf.name})")

assert all(p.exists() and p.stat().st_size > 0 for p in figs)
assert all(pdf.exists() and pdf.stat().st_size > 0 for pdf in pdfs)

for name in ("study_area_hwm", "sensitivity_summary", "susceptibility_maps_comparison"):
    display(Image(filename=str(report.FIGURES_DIR / f"{name}.png")))

required = {"study_area_hwm", "susceptibility_maps", "jackknife_importance",
            "internal_validation_curves", "gfi_comparison", "sensitivity_summary"}
assert required <= {p.stem for p in figs}
assert "susceptibility_maps_comparison" in {p.stem for p in figs}
print(f"gate: {len(figs)} figures (PNG+PDF pairs, 300 DPI) under reports/figures/ -- "
      f"all {len(required)} necessary paper figures present, INCLUDING the 4-model "
      f"susceptibility_maps_comparison map (rendered from Stage 4's "
      f"model_compare.susceptibility_rasters() output -- no longer skipped) -- OK")
